# NB3: Interpretation & Validation (slim)

NB2 의 best multi-output 메타러너에 대해 세 가지:
1. 어떤 feature 가 중요했는가 — `perf_score` / `time_score` / `mem_score` 각각의 native gain importance
2. 선택한 데이터셋 1개에서 48 combo 의 score 분포 (메타피처 vs target)
3. **신규 Milano 지역 (예: Milano-8815) 추론 + 실제 BrickTS 결과 비교** — 메타러너의 일반화 성능을 holdout 평가

**제거 (archieve 대비)**:
- §1b 대표 트리 시각화 (multi-output LGB/XGB 의 단일 트리 시각화는 직관적 의미 약함)
- 외부 벤치마크 ranking sync (NB2 §10 에서 raw 비교 표로 격하)

**입력**: NB1/NB2 출력

**출력** (`outputs/nb3/`):
- `feature_importance_best_multitarget.csv`
- `combo_view_single_dataset.csv`
- `inference_test_results.csv`, `inference_test_metrics.csv`


## Section 0 — Setup + NB1/NB2 출력 로드

In [ ]:
%pip install --quiet numpy pandas scipy scikit-learn matplotlib seaborn tqdm xgboost lightgbm statsmodels pycatch22 antropy nolds PyWavelets EntropyHub

In [ ]:
from __future__ import annotations
import glob, json, re, pickle, warnings
from pathlib import Path
from typing import Optional
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

NB1, NB2 = Path('outputs/nb1'), Path('outputs/nb2')
OUT = Path('outputs/nb3'); OUT.mkdir(parents=True, exist_ok=True)

dataset_meta    = pd.read_csv(NB1 / 'dataset_metadata.csv').set_index('dataset')
meta_full       = pd.read_csv(NB1 / 'meta_features_full.csv').set_index('dataset')
feature_groups  = pd.read_csv(NB1 / 'feature_groups.csv')
brickts_wide    = pd.read_csv(NB2 / 'brickts_wide.csv')
fi_tree         = pd.read_csv(NB2 / 'feature_importance_tree.csv')

with open(NB2 / 'best_model.pkl', 'rb') as f:
    bundle = pickle.load(f)
BEST_MODEL_NAME = bundle['model_name']
TARGET_COLS     = bundle['targets']
PRIMARY_TARGET  = bundle['primary_target']
FEATURE_LIST    = bundle['feature_list']
META_FEATS      = bundle['selected_meta_features']
ONEHOT_COLS     = bundle['onehot_cols']
TRAIN_DATASETS  = bundle.get('train_datasets', [])
VAL_DATASETS    = bundle.get('val_datasets', [])
TEST_DATASETS   = bundle.get('test_datasets', bundle.get('milano_holdout', []))
BM              = bundle['model']
LABELS          = ['CD', 'CI']

print(f'Model: {BEST_MODEL_NAME}  |  targets = {TARGET_COLS}  |  primary = {PRIMARY_TARGET}')
print(f'TRAIN={len(TRAIN_DATASETS)}  VAL={len(VAL_DATASETS)}  TEST={len(TEST_DATASETS)}')
print(f'#features: {len(FEATURE_LIST)}  (meta={len(META_FEATS)}, one-hot={len(FEATURE_LIST)-len(META_FEATS)})')


## Section 1 — Feature Importance (target 별)

NB2 §8 의 `feature_importance_tree.csv` 에서 target 별 top-N / bottom-N (`TOP_N_FEAT`, 기본 10) panel.


In [ ]:
fi_best = fi_tree[fi_tree['model'] == BEST_MODEL_NAME].copy()
fi_best.to_csv(OUT / 'feature_importance_best_multitarget.csv', index=False)

TOP_N_FEAT = 10
fig, axes = plt.subplots(2, 3, figsize=(20, 11))
for col, tgt in enumerate(TARGET_COLS):
    sub = fi_best[fi_best['target'] == tgt]
    top = sub.nlargest(TOP_N_FEAT, 'importance')
    bot = sub.nsmallest(TOP_N_FEAT, 'importance')
    sns.barplot(data=top, y='feature', x='importance', ax=axes[0, col], color='steelblue')
    axes[0, col].set_title(f'Top-{TOP_N_FEAT} — {tgt}')
    sns.barplot(data=bot, y='feature', x='importance', ax=axes[1, col], color='salmon')
    axes[1, col].set_title(f'Bottom-{TOP_N_FEAT} — {tgt}')
plt.suptitle(f'{BEST_MODEL_NAME} native importance per target', y=1.005)
plt.tight_layout(); plt.show()
fi_best.head()


In [ ]:
# === [논문 Figure] Feature Importance 그룹 요약 표 ===
# row: Dataset(메타피처 그룹) / Model(arch·scope·level),  col: avg rate / avg rank / Top-K hit / Bottom-K hit
from IPython.display import display
FI_TARGET = PRIMARY_TARGET          # 집계할 target (기본 perf_score). 'time_score'/'mem_score' 로 변경 가능
FI_TOPK   = 10                      # Top-K / Bottom-K hit 기준 (사용자 설정)

GROUP_NAMES = {                     # 메타피처 그룹 → 논문용 이름
    'A': 'Target complexity',
    'B': 'Target memory/scale',
    'C': 'Target decomposition/freq',
    'Catch22': 'Catch22',
    'D': 'Cross-channel relation',
    'E': 'Distribution shift',
}
_feat2group = feature_groups.set_index('feature')['group'].to_dict()

def _feat_group(feat):
    """feature -> (super, subgroup). super in {Dataset, Model}."""
    if feat.startswith('arch_'):  return ('Model', 'arch')
    if feat.startswith('scope_'): return ('Model', 'scope')
    if feat.startswith('level_'): return ('Model', 'level')
    if feat.startswith('c22_'):   return ('Dataset', GROUP_NAMES['Catch22'])
    g = _feat2group.get(feat, '?')
    return ('Dataset', GROUP_NAMES.get(g, g))

sub = fi_best[fi_best['target'] == FI_TARGET].copy()
if len(sub) == 0:
    print(f'[!] target {FI_TARGET} 없음. 가능: {sorted(fi_best["target"].unique())}')
else:
    tot = sub['importance'].sum()
    sub['rate'] = sub['importance'] / tot if tot > 0 else 0.0
    N = len(sub)
    sub['is_top'] = sub['rank'] <= FI_TOPK
    sub['is_bot'] = sub['rank'] > (N - FI_TOPK)
    sub[['super', 'subgroup']] = sub['feature'].apply(lambda f: pd.Series(_feat_group(f)))

    agg = (sub.groupby(['super', 'subgroup'])
              .agg(avg_rate=('rate', 'mean'),
                   avg_rank=('rank', 'mean'),
                   topk=('is_top', 'sum'),
                   botk=('is_bot', 'sum'),
                   n_feat=('feature', 'count'))
              .reset_index())
    agg['_o'] = agg['super'].map({'Dataset': 0, 'Model': 1})
    agg = agg.sort_values(['_o', 'avg_rank']).drop(columns='_o')
    agg['avg_rate'] = (agg['avg_rate'] * 100).round(2)     # %
    agg['avg_rank'] = agg['avg_rank'].round(1)
    fi_table = agg.set_index(['super', 'subgroup'])
    fi_table.columns = ['avg_rate(%)', 'avg_rank', f'Top{FI_TOPK}_hit', f'Bottom{FI_TOPK}_hit', 'n_feat']
    fi_table.to_csv(OUT / 'fi_group_table.csv')
    print(f'Feature importance group table  (target={FI_TARGET}, K={FI_TOPK}, N_features={N})')
    display(fi_table)

    # 논문용 matplotlib 표 렌더
    cell_text = [[str(a), str(b)] + [f'{v}' for v in row]
                 for (a, b), row in zip(fi_table.index, fi_table.values)]
    fig, ax = plt.subplots(figsize=(11, 0.55 * len(fi_table) + 1.3)); ax.axis('off')
    tbl = ax.table(cellText=cell_text, colLabels=['super', 'subgroup'] + list(fi_table.columns),
                   loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.45)
    ax.set_title(f'Feature Importance by group — target={FI_TARGET}  (K={FI_TOPK})', pad=14)
    plt.tight_layout(); plt.show()


## Section 1B — BrickTS 결과 요약 표 (논문 Figure)

row: (arch | scope | level) → 세부분류 → (CD / CI),  column: (Milano | Other domain) → dataset.
값은 선택 metric(`SUMMARY_METRIC`, 기본 `mse`)의 평균. 시각화 dataset 은 `MILANO_VIZ`/`OTHER_VIZ` 로 설정.

In [ ]:
# === [논문 Figure] BrickTS 결과 요약 표 ===
from IPython.display import display
SUMMARY_METRIC = 'mse'      # 'mse' | 'train_time' | 'num_params'  (CD/CI 평균값)
MILANO_VIZ = ['Milano-6610', 'Milano-6373', 'Milano-6265', 'Milano-4275', 'Milano-6165', 'Milano-7108']
OTHER_VIZ  = ['ETTh1', 'ILI', 'Traffic', 'Exchange', 'Weather']

_axes = {
    'arch':  ['mlp', 'rnn', 'cnn', 'transformer'],
    'scope': ['global', 'local', 'hierarchical', 'sparse'],
    'level': ['direct', 'decomposition', 'spectral'],
}
_cd, _ci = f'{SUMMARY_METRIC}_CD', f'{SUMMARY_METRIC}_CI'
assert _cd in brickts_wide.columns, f'{_cd} 없음 — SUMMARY_METRIC 후보: mse / train_time / num_params'

# long: dataset, arch, scope, level, strategy, value
_parts = []
for strat, col in [('CD', _cd), ('CI', _ci)]:
    t = brickts_wide[['dataset', 'arch', 'scope', 'level', col]].rename(columns={col: 'value'}).copy()
    t['strategy'] = strat
    _parts.append(t)
bw_long = pd.concat(_parts, ignore_index=True)

# column(dataset) — 존재하는 것만, 도메인 라벨 부여
_spec = [('Milano', d) for d in MILANO_VIZ] + [('Other', d) for d in OTHER_VIZ]
_present = set(bw_long['dataset'].unique())
_cols = [(dom, d) for dom, d in _spec if d in _present]
_missing = [d for dom, d in _spec if d not in _present]
if _missing:
    print('[!] brickts_wide 에 없는 dataset 제외:', _missing)
ds2dom = {d: dom for dom, d in _cols}
viz_ds = [d for dom, d in _cols]

# row (axis, subcat, strategy) x dataset 평균
index_tuples, records = [], []
for axis, subcats in _axes.items():
    for sc in subcats:
        if sc not in set(bw_long[axis].unique()):
            continue
        for strat in ['CD', 'CI']:
            sel = bw_long[(bw_long[axis] == sc) & (bw_long['strategy'] == strat) &
                          (bw_long['dataset'].isin(viz_ds))]
            means = sel.groupby('dataset')['value'].mean()
            index_tuples.append((axis, sc, strat))
            records.append([means.get(d, np.nan) for d in viz_ds])

brickts_summary = pd.DataFrame(
    records,
    index=pd.MultiIndex.from_tuples(index_tuples, names=['axis', 'subcat', 'strategy']),
    columns=pd.MultiIndex.from_tuples([(ds2dom[d], d) for d in viz_ds], names=['domain', 'dataset']),
).round(4)
brickts_summary.to_csv(OUT / 'brickts_summary_table.csv')
print(f'BrickTS summary — {SUMMARY_METRIC} (CD/CI mean over other axes)')
display(brickts_summary)


## Section 2 — Feature → score 단일 데이터셋 시각화 (48 combo)

선택한 데이터셋 1개에 대해 48 combo 의 score (perf/time/params) 분포 확인.
- X = 선택한 메타피처 값 (데이터셋 내 단일 값 + jitter)
- Y = `VIEW_TARGET` ∈ {`perf_score`, `time_score`, `mem_score`}
- Color = USER 선택 axis (`arch` | `scope` | `level`)
- 보조선: `y=0` (CD/CI 경계, score<0 → CD / score>=0 → CI)


In [ ]:
# ===== USER CONFIG =====
VIEW_DATASET  = 'ETTh1'                  # 메타피처 출처 데이터셋 1개
VIEW_FEATURES = ['trend_strength_ot']    # 1+ meta features (각각 panel)
VIEW_TARGET   = 'perf_score'             # 'perf_score' | 'time_score' | 'mem_score'
COLOR_AXIS    = 'arch'                    # 'arch' | 'scope' | 'level'
# =======================
print(f'VIEW_DATASET={VIEW_DATASET}  |  VIEW_FEATURES={VIEW_FEATURES}')
print(f'VIEW_TARGET={VIEW_TARGET}  |  COLOR_AXIS={COLOR_AXIS}')
print()
print('VIEW_DATASET candidates:', list(meta_full.index))
print('VIEW_TARGET   candidates:', TARGET_COLS)
print('COLOR_AXIS    candidates: ["arch", "scope", "level"]')


In [ ]:
sub = brickts_wide.query('dataset == @VIEW_DATASET').copy()
if len(sub) == 0:
    raise ValueError(f"No BrickTS rows for dataset '{VIEW_DATASET}'")

ds_features = meta_full.loc[VIEW_DATASET]
rng = np.random.default_rng(0)

n = len(VIEW_FEATURES)
fig, axes = plt.subplots(1, n, figsize=(6.5 * n, 6), squeeze=False)
for ax, feat in zip(axes[0], VIEW_FEATURES):
    x_val = float(ds_features.get(feat, np.nan))
    spread = max(abs(x_val) * 0.01, 0.01)
    sub_plot = sub.copy()
    sub_plot['_x'] = x_val + rng.normal(0, spread, len(sub_plot))
    sns.scatterplot(data=sub_plot, x='_x', y=VIEW_TARGET, hue=COLOR_AXIS,
                    s=110, alpha=0.85, edgecolor='black', ax=ax, legend='brief')
    ax.axhline(0, color='black', linestyle='--', alpha=0.6, label='_nolegend_')
    ax.axvline(x_val, color='gray', linestyle='--', alpha=0.4, label='_nolegend_')
    ax.set_xlabel(f'{feat}  ({VIEW_DATASET} = {x_val:.4f})')
    ax.set_ylabel(f'{VIEW_TARGET}   (+1 = CI win, -1 = CD win)')
    ax.set_title(f'{VIEW_DATASET}  |  color = {COLOR_AXIS}')
    ax.legend(title=COLOR_AXIS, fontsize=8, loc='lower left')
plt.tight_layout(); plt.show()

out = sub[['arch', 'scope', 'level'] + TARGET_COLS].copy()
for f in VIEW_FEATURES:
    out[f] = float(ds_features.get(f, np.nan))
out['view_dataset'] = VIEW_DATASET
out['view_target']  = VIEW_TARGET
out.to_csv(OUT / 'combo_view_single_dataset.csv', index=False)
out.head()


## Section 3 — 신규 지역 추론 + 실제 BrickTS 결과 비교

`INFERENCE_DATASETS` 의 각 데이터셋 (예: Milano-8815) 에 대해:
1. NB1 메타피처 (`meta_full`) 에서 해당 dataset 1 행 추출
2. 메타러너가 48 (arch, scope, level) combo 의 (perf, time, params) score 예측
3. `results_BrickTS/{folder}/.../*_results.json` 에서 실제 BrickTS 결과 로드 → `to_wide_with_scores` 로 (true) score 산출
4. 예측 vs 실제 (per target): **MAE, Spearman ρ, sign_acc, 2-class(CD/CI) f1 + accuracy**, top-K 추천 일치도
5. PRIMARY_TARGET(perf_score) 상위 5개 모델조합 — 실제 BrickTS vs 예측 비교 (combo overlap)
6. PRIMARY_TARGET 기준 2-class(CD/CI) confusion matrix

이 세션이 메타러너의 **진짜 holdout 일반화 검증**.


In [ ]:
# NB2 의 BrickTS 결과 로딩 함수 재정의 (notebook self-contained)
ARCH_TYPES  = ['mlp', 'rnn', 'cnn', 'transformer']
SCOPE_TYPES = ['global', 'local', 'hierarchical', 'sparse']
LEVEL_TYPES = ['direct', 'decomposition', 'spectral']
STRATEGIES  = ['CD', 'CI']

# 표준 벤치마크 폴더만 수동 매핑. Milano-<K> 는 results_BrickTS 스캔으로 자동 등록 (register_milano_folders).
FOLDER_TO_NAME = {
    'ETTh1': 'ETTh1', 'ETTh2': 'ETTh2', 'ETTm1': 'ETTm1', 'ETTm2': 'ETTm2',
    'traffic': 'Traffic', 'electricity': 'Electricity',
    'exchange_rate': 'Exchange', 'exchange': 'Exchange',
    'weather': 'Weather', 'national_illness': 'ILI', 'ILI': 'ILI',
}

def _folder_to_name(folder: str) -> str:
    """폴더명 → 데이터셋 표준명. milano_<K> / milano-<K> 는 자동으로 Milano-<K>."""
    if folder in FOLDER_TO_NAME:
        return FOLDER_TO_NAME[folder]
    m = re.match(r'^milano[_-](\d+)$', folder, re.I)
    if m:
        return f'Milano-{m.group(1)}'
    return folder


RESULTS_BRICKTS_CANDIDATES = ['../results_BrickTS', 'results_BrickTS', './results_BrickTS']


def register_milano_folders(results_dir_candidates) -> list:
    """results_BrickTS 하위 폴더를 스캔해 모든 milano_<K> 를 FOLDER_TO_NAME 에 자동 등록."""
    found = []
    for base in results_dir_candidates:
        bp = Path(base)
        if not bp.is_dir():
            continue
        for sub in bp.iterdir():
            if not sub.is_dir():
                continue
            mm = re.match(r'^milano[_-](\d+)$', sub.name, re.I)
            if mm:
                std = f'Milano-{mm.group(1)}'
                FOLDER_TO_NAME[sub.name] = std
                FOLDER_TO_NAME[std] = std
                found.append(std)
    return sorted(set(found), key=lambda s: int(s.split('-')[1]))


_milano_registered = register_milano_folders(RESULTS_BRICKTS_CANDIDATES)
print(f'Auto-registered {len(_milano_registered)} Milano folders to FOLDER_TO_NAME: {_milano_registered}')


TRAIN_TIME_KEYS = ('train_time_mean', 'train_time', 'training_time', 'training_time_seconds')


def _scal_train_time(scal: dict) -> float:
    for k in TRAIN_TIME_KEYS:
        v = scal.get(k)
        if v is not None:
            return float(v)
    return float('nan')


def _ot_metrics_from_payload(data, mode, ot_name):
    ch = data.get('per_channel', {}).get(ot_name) if mode == 'CD' else data.get(ot_name)
    if not ch:
        return (np.nan, np.nan, np.nan, np.nan)
    scal = (data.get('scalability') if mode == 'CD' else ch.get('scalability')) or {}
    return (float(ch.get('mse', np.nan)),
            float(ch.get('mae', np.nan)),
            float(scal.get('num_params', np.nan)),
            _scal_train_time(scal))


def _parse_brickts_path(path):
    parts = path.parts
    if 'results_BrickTS' not in parts:
        return None
    ri = parts.index('results_BrickTS')
    if ri + 4 >= len(parts):
        return None
    combo = parts[ri + 4]
    stem = path.stem.replace('_results', '')
    mode = None
    if stem.startswith(combo + '_'):
        rest = stem[len(combo) + 1:].split('_')
        if rest and rest[0] in STRATEGIES:
            mode = rest[0]
    if mode is None:
        _, tail = stem.rsplit('_', 1)
        mode = tail if tail in STRATEGIES else None
    if mode is None:
        return None
    axis = combo.split('_')
    if len(axis) < 3 or axis[0] not in ARCH_TYPES or axis[1] not in SCOPE_TYPES:
        return None
    level = '_'.join(axis[2:])
    if level not in LEVEL_TYPES:
        return None
    return {
        'dataset': _folder_to_name(parts[ri + 1]),
        'arch': axis[0], 'scope': axis[1], 'level': level,
        'strategy': mode,
    }


def load_brickts_long(results_dir, ot_name_by_dataset):
    rows = []
    for fp in glob.glob(f'{results_dir}/**/*_results.json', recursive=True):
        meta = _parse_brickts_path(Path(fp))
        if meta is None:
            continue
        with open(fp) as f:
            data = json.load(f)
        ot_name = ot_name_by_dataset.get(meta['dataset'], 'OT')
        mse, mae, n_params, t = _ot_metrics_from_payload(data, meta['strategy'], ot_name)
        rows.append({**meta, 'mse': mse, 'mae': mae, 'num_params': n_params, 'train_time': t})
    cols = ['dataset', 'arch', 'scope', 'level', 'strategy', 'mse', 'mae', 'num_params', 'train_time']
    return pd.DataFrame(rows, columns=cols)


def _signed_diff_core(cd, ci):
    denom = np.maximum(cd.values, ci.values)
    out = np.full_like(denom, np.nan, dtype=float)
    mask = (denom > 0) & np.isfinite(cd) & np.isfinite(ci)
    out[mask] = (cd.values[mask] - ci.values[mask]) / denom[mask]
    return pd.Series(out, index=cd.index)


def compute_perf_score(mse_cd, mse_ci):
    """MSE 기반 성능 score (results_viz.create_performance_heatmap 공식)."""
    return _signed_diff_core(mse_cd, mse_ci)


def compute_efficiency_score(cost_cd, cost_ci):
    """Time/Memory 기반 효율 score (results_viz.create_efficiency_comparison_heatmap 공식)."""
    return _signed_diff_core(cost_cd, cost_ci)


# 하위 호환
signed_diff = compute_perf_score


REQUIRED_WIDE_COLS = ['mse_CD', 'mse_CI',
                       'train_time_CD', 'train_time_CI',
                       'num_params_CD', 'num_params_CI']


def to_wide_with_scores(long_df):
    wide = long_df.pivot_table(
        index=['dataset', 'arch', 'scope', 'level'],
        columns='strategy',
        values=['mse', 'train_time', 'num_params'],
    ).reset_index()
    wide.columns = [f'{a}_{b}' if b else a for a, b in wide.columns]
    for col in REQUIRED_WIDE_COLS:
        if col not in wide.columns:
            wide[col] = np.nan
    wide = wide.dropna(subset=['mse_CD', 'mse_CI'])
    wide['perf_score']   = compute_perf_score(wide['mse_CD'],        wide['mse_CI'])
    wide['time_score']   = compute_efficiency_score(wide['train_time_CD'], wide['train_time_CI'])
    wide['mem_score'] = compute_efficiency_score(wide['num_params_CD'], wide['num_params_CI'])
    return wide


# 2-class(CD/CI) + per-target metrics (NB2 와 동일)
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, f1_score, accuracy_score, confusion_matrix, roc_auc_score


def to_two_class(y):
    return np.where(y >= 0, 'CI', 'CD')


def per_target_metrics(y_true, y_pred):
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[m], y_pred[m]
    if len(y_true) < 2:
        nan = float('nan')
        return {'mse': nan, 'mae': nan, 'spearman': nan,
                'sign_acc': nan, 'accuracy': nan, 'f1': nan, 'auc': nan, 'n': int(m.sum())}
    sp, _ = spearmanr(y_true, y_pred)
    cls_true = to_two_class(y_true)
    cls_pred = to_two_class(y_pred)
    labels = ['CD', 'CI']
    yt_bin = (y_true >= 0).astype(int)   # CI=1, CD=0
    try:
        auc = float(roc_auc_score(yt_bin, y_pred)) if len(np.unique(yt_bin)) == 2 else float('nan')
    except ValueError:
        auc = float('nan')
    return {
        'mse':       float(mean_squared_error(y_true, y_pred)),
        'mae':       float(mean_absolute_error(y_true, y_pred)),
        'spearman':  float(sp) if not np.isnan(sp) else 0.0,
        'sign_acc':  float(np.mean(np.sign(y_true) == np.sign(y_pred))),
        'accuracy':  float(accuracy_score(cls_true, cls_pred)),
        'f1':        float(f1_score(cls_true, cls_pred, average='macro',
                                     labels=labels, zero_division=0)),
        'auc':       auc,
        'n': int(len(y_true)),
    }


In [ ]:
def _build_48_combo_matrix(feat_vec: dict) -> pd.DataFrame:
    """4 arch × 4 scope × 3 level = 48 rows. 한 dataset 의 메타피처 vector 를 모든 combo 에 broadcast."""
    rows = []
    for a in ARCH_TYPES:
        for s in SCOPE_TYPES:
            for l in LEVEL_TYPES:
                row = {c: 0 for c in FEATURE_LIST}
                for f in META_FEATS:
                    row[f] = feat_vec.get(f, np.nan)
                row[f'arch_{a}']  = 1
                row[f'scope_{s}'] = 1
                row[f'level_{l}'] = 1
                row.update({'arch': a, 'scope': s, 'level': l})
                rows.append(row)
    return pd.DataFrame(rows)


def predict_combos_for_dataset(dataset_name: str) -> pd.DataFrame:
    feat_vec = meta_full.loc[dataset_name].to_dict()
    df = _build_48_combo_matrix(feat_vec)
    # train-time 분포로 NaN 메타피처 fill
    for col in META_FEATS:
        if df[col].isna().all():
            df[col] = float(meta_full[col].median()) if col in meta_full.columns else 0.0
    Y = np.asarray(BM.predict(df[FEATURE_LIST].astype(float)), dtype=float)
    df[f'pred_{TARGET_COLS[0]}'] = Y[:, 0]
    df[f'pred_{TARGET_COLS[1]}'] = Y[:, 1]
    df[f'pred_{TARGET_COLS[2]}'] = Y[:, 2]
    return df[['arch', 'scope', 'level',
                f'pred_{TARGET_COLS[0]}',
                f'pred_{TARGET_COLS[1]}',
                f'pred_{TARGET_COLS[2]}']]


def load_actual_brickts_wide(dataset_name: str, results_dir_candidates=None) -> Optional[pd.DataFrame]:
    if results_dir_candidates is None:
        results_dir_candidates = ['../results_BrickTS', 'results_BrickTS', './results_BrickTS']
    ot_map = dataset_meta['ot_name'].to_dict()
    for cand in results_dir_candidates:
        long_df = load_brickts_long(cand, ot_map)
        if long_df.empty or 'dataset' not in long_df.columns:
            continue
        sub = long_df[long_df['dataset'] == dataset_name]
        if len(sub):
            wide = to_wide_with_scores(sub)
            return wide[['arch', 'scope', 'level',
                          'perf_score', 'time_score', 'mem_score']]
    return None


def evaluate_inference(dataset_name: str, top_k: int = 5, weights: dict = None):
    pred = predict_combos_for_dataset(dataset_name)
    actual = load_actual_brickts_wide(dataset_name)
    if actual is None or len(actual) == 0:
        print(f'[{dataset_name}] BrickTS 실제 결과 없음 — 예측만 반환')
        return pred, None, None

    merged = pred.merge(actual, on=['arch', 'scope', 'level'], how='inner')
    if len(merged) == 0:
        print(f'[{dataset_name}] BrickTS 실제 결과 매칭 0건')
        return pred, None, None

    rows = []
    for tgt in TARGET_COLS:
        m = per_target_metrics(merged[tgt].values, merged[f'pred_{tgt}'].values)
        rows.append({'dataset': dataset_name, 'target': tgt, **m})

    # Top-K 추천 일치도 (perf_score 기준; +1 = CI 좋음에 가까운 combo 상위)
    w = weights or {'perf': 1.0, 'time': 0.0, 'mem': 0.0}
    merged['pred_final'] = (w.get('perf', 1.0)  * merged[f'pred_{TARGET_COLS[0]}']
                            + w.get('time', 0.0)   * merged[f'pred_{TARGET_COLS[1]}']
                            + w.get('mem', 0.0) * merged[f'pred_{TARGET_COLS[2]}'])
    merged['actual_final'] = (w.get('perf', 1.0)  * merged[TARGET_COLS[0]]
                              + w.get('time', 0.0)   * merged[TARGET_COLS[1]]
                              + w.get('mem', 0.0) * merged[TARGET_COLS[2]])
    # |final| 큰 순서 (확신이 큰 combo) top-K
    pred_topk   = set(merged.reindex(merged['pred_final'].abs().sort_values(ascending=False).index)
                       .head(top_k).set_index(['arch','scope','level']).index)
    actual_topk = set(merged.reindex(merged['actual_final'].abs().sort_values(ascending=False).index)
                       .head(top_k).set_index(['arch','scope','level']).index)
    topk_overlap = len(pred_topk & actual_topk)
    metrics_df = pd.DataFrame(rows)
    metrics_df['top_k'] = top_k
    metrics_df['topk_overlap'] = topk_overlap
    merged['dataset'] = dataset_name
    return pred, merged, metrics_df


In [ ]:
# ===== USER CONFIG (Section 3) =====
# inference 대상: NB2 가 분리한 test 지역 (train+val 과 절대 겹치지 않음).
# None = test 전부.  리스트로 일부만 지정 가능 (VIZ_DATASETS 처럼).
MILANO_INFER_DATASETS = None   # 예: ['Milano-6610', 'Milano-6373', 'Milano-6265', 'Milano-4275', 'Milano-6165', 'Milano-7108']
INCLUDE_STANDARD_IN_INFERENCE = True    # 표준 벤치마크도 inference 검증에 포함
STANDARD_DATASETS = ['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2',
                     'Traffic', 'Electricity', 'Exchange', 'Weather', 'ILI']
WEIGHTS           = {'perf': 1.0, 'time': 0.0, 'mem': 0.0}
TOP_K             = 5

_milano_pat = re.compile(r'^Milano-\d+$')
_excluded = set(TRAIN_DATASETS) | set(VAL_DATASETS)   # 학습/검증 지역은 inference 금지
_bw = set(brickts_wide['dataset'].unique())
def _avail(d):
    return d in _bw and d in meta_full.index and d not in _excluded

_holdout = [d for d in TEST_DATASETS if _milano_pat.match(str(d)) and _avail(d)]
if MILANO_INFER_DATASETS is not None:
    _sel = set(MILANO_INFER_DATASETS)
    _miss = [d for d in MILANO_INFER_DATASETS if d not in set(_holdout)]
    if _miss:
        print('[!] MILANO_INFER_DATASETS 중 test/holdout 에 없어 무시:', _miss)
    _holdout = [d for d in _holdout if d in _sel]
_holdout = sorted(set(_holdout), key=lambda s: int(s.split('-')[1]))

INFERENCE_DATASETS = list(_holdout)
_std_inf = []
if INCLUDE_STANDARD_IN_INFERENCE:
    _std_inf = [d for d in STANDARD_DATASETS if _avail(d)]
    INFERENCE_DATASETS = INFERENCE_DATASETS + _std_inf

print(f'Inference Milano/test (n={len(_holdout)}) : {_holdout}')
print(f'Standard in inference (n={len(_std_inf)}) : {_std_inf}')
print(f'Excluded train+val (n={len(_excluded)})')
print(f'INFERENCE_DATASETS (n={len(INFERENCE_DATASETS)}) : {INFERENCE_DATASETS}')
# ====================================

all_merged, all_metrics = [], []
for ds in INFERENCE_DATASETS:
    if ds not in meta_full.index:
        print(f'[{ds}] NB1 meta_features_full.csv 에 해당 dataset 없음 — skip')
        continue
    pred, merged, metrics = evaluate_inference(ds, top_k=TOP_K, weights=WEIGHTS)
    if merged is not None:
        all_merged.append(merged)
    if metrics is not None:
        all_metrics.append(metrics)

inf_results = pd.concat(all_merged, ignore_index=True) if all_merged else pd.DataFrame()
inf_metrics = pd.concat(all_metrics, ignore_index=True) if all_metrics else pd.DataFrame()
inf_results.to_csv(OUT / 'inference_test_results.csv', index=False)
inf_metrics.to_csv(OUT / 'inference_test_metrics.csv', index=False)

print('Per-target metrics by dataset:')
if len(inf_metrics):
    print(inf_metrics[['dataset', 'target', 'n', 'accuracy', 'f1', 'auc', 'topk_overlap']].round(3).to_string(index=False))

    # target 별 평균 (테스트한 INFERENCE_DATASETS 전체)
    avg_cols = ['accuracy', 'f1', 'auc', 'topk_overlap']
    avg_by_target = inf_metrics.groupby('target')[avg_cols].mean().reindex(TARGET_COLS).round(3)
    avg_by_target.insert(0, 'n_datasets', inf_metrics.groupby('target').size().reindex(TARGET_COLS))
    avg_by_target.to_csv(OUT / 'inference_test_metrics_avg.csv')
    print()
    print('Per-target average over tested datasets:')
    print(avg_by_target.to_string())
else:
    print('(no metrics — BrickTS 실제 결과 없는 듯)')


In [ ]:
# 실제(BrickTS) vs 예측 — PRIMARY_TARGET(perf_score) |절댓값| 상위 5개 모델조합 비교 (데이터셋별)
# |score| 가 큰 = CD/CI 우열이 뚜렷한 combo. topk_overlap 과 동일하게 절댓값 기준으로 정렬.
if len(inf_results):
    TOP_N = 5
    pcol, ppred = PRIMARY_TARGET, f'pred_{PRIMARY_TARGET}'
    top_rows = []
    for ds in inf_results['dataset'].unique():
        sub = inf_results[inf_results['dataset'] == ds].copy()
        sub['combo'] = sub['arch'] + '_' + sub['scope'] + '_' + sub['level']
        actual_top = (sub.reindex(sub[pcol].abs().sort_values(ascending=False).index)
                         .head(TOP_N)[['combo', pcol]].reset_index(drop=True))
        pred_top = (sub.reindex(sub[ppred].abs().sort_values(ascending=False).index)
                       .head(TOP_N)[['combo', ppred]].reset_index(drop=True))
        overlap = len(set(actual_top['combo']) & set(pred_top['combo']))
        cmp = pd.concat([
            actual_top.rename(columns={'combo': 'actual_combo', pcol: 'actual_perf'}),
            pred_top.rename(columns={'combo': 'pred_combo', ppred: 'pred_perf'}),
        ], axis=1)
        cmp.index = [f'top{i+1}' for i in range(len(cmp))]
        print(f'\n[{ds}]  {PRIMARY_TARGET} |abs| Top-{TOP_N}  (actual vs predicted)  |  combo overlap = {overlap}/{TOP_N}')
        print(cmp.round(3).to_string())
        rec = cmp.reset_index().rename(columns={'index': 'rank'})
        rec.insert(0, 'dataset', ds)
        rec['overlap'] = overlap
        top_rows.append(rec)
    perf_top_cmp = pd.concat(top_rows, ignore_index=True)
    perf_top_cmp.to_csv(OUT / 'perf_top5_actual_vs_pred.csv', index=False)
    print(f'\nMean |abs| Top-{TOP_N} combo overlap over tested datasets: '
          f"{perf_top_cmp.groupby('dataset')['overlap'].first().mean():.2f}/{TOP_N}")
else:
    print('(no inference results — BrickTS 실제 결과 없는 듯)')


In [ ]:
# PRIMARY_TARGET 2-class(CD/CI) confusion matrix — 선택한 데이터셋만
# ===== 시각화 대상 선택 =====
if len(inf_results):
    print('Available inference datasets:', inf_results['dataset'].unique().tolist())
VIZ_DATASETS = None   # None = 전체. 예: ['Milano-1234', 'ETTh1'] 로 일부만 시각화
# ===========================

if len(inf_results):
    _all_ds = inf_results['dataset'].unique().tolist()
    VIZ_DS = [d for d in (VIZ_DATASETS or _all_ds) if d in _all_ds]
    _miss = [d for d in (VIZ_DATASETS or []) if d not in _all_ds]
    if _miss:
        print('[!] inf_results 에 없는 데이터셋 무시:', _miss)
    n = len(VIZ_DS)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4.5), squeeze=False)
    for ax, ds in zip(axes[0], VIZ_DS):
        sub = inf_results[inf_results['dataset'] == ds]
        yt = to_two_class(sub[PRIMARY_TARGET].values)
        yp = to_two_class(sub[f'pred_{PRIMARY_TARGET}'].values)
        cm = confusion_matrix(yt, yp, labels=LABELS)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
                    xticklabels=LABELS, yticklabels=LABELS, ax=ax)
        ax.set_xlabel('Predicted'); ax.set_ylabel('Actual (BrickTS)')
        f1 = f1_score(yt, yp, average='macro', labels=LABELS, zero_division=0)
        acc = accuracy_score(yt, yp)
        ax.set_title(f'{ds}  |  {PRIMARY_TARGET}\nF1 = {f1:.3f}  acc = {acc:.3f}')
    plt.suptitle('Inference — 실제 BrickTS vs 메타러너 예측 (선택 데이터셋)', y=1.02)
    plt.tight_layout(); plt.show()
else:
    print('(no inference results)')


In [ ]:
# AUC(ROC) 시각화.
#  - 개별 ROC 곡선: VIZ_DATASETS 로 설정된 데이터셋만
#  - pooled ROC: VIZ 설정과 무관하게 Test 데이터셋 전부 (inf_results 내 test)
#  - legend 는 완전히 분리(별도 Figure), AUC 막대도 별도 Figure
from sklearn.metrics import roc_curve
if 'VIZ_DATASETS' not in globals():
    VIZ_DATASETS = None

def _roc_xy(df_sub):
    mask = np.isfinite(df_sub[PRIMARY_TARGET]) & np.isfinite(df_sub[f'pred_{PRIMARY_TARGET}'])
    yt = (df_sub.loc[mask, PRIMARY_TARGET].values >= 0).astype(int)   # CI=1, CD=0
    ys = df_sub.loc[mask, f'pred_{PRIMARY_TARGET}'].values
    return yt, ys

if len(inf_results):
    _all_ds = inf_results['dataset'].unique().tolist()
    VIZ_DS = [d for d in (VIZ_DATASETS or _all_ds) if d in _all_ds]
    TEST_IN_RESULTS = [d for d in _all_ds if d in set(TEST_DATASETS)]   # pooled 대상: test 전부

    # ===== Figure 1: ROC 곡선 (개별=VIZ_DS, pooled=test 전부) — 곡선만, legend 없음 =====
    fig, ax = plt.subplots(figsize=(6.2, 5.8))
    for ds in VIZ_DS:
        yt, ys = _roc_xy(inf_results[inf_results['dataset'] == ds])
        if len(np.unique(yt)) == 2:
            fpr, tpr, _ = roc_curve(yt, ys)
            ax.plot(fpr, tpr, lw=1.6, alpha=0.85, label=f'{ds} (AUC={roc_auc_score(yt, ys):.3f})')
    p_yt, p_ys = [], []
    for ds in TEST_IN_RESULTS:
        yt, ys = _roc_xy(inf_results[inf_results['dataset'] == ds])
        p_yt.append(yt); p_ys.append(ys)
    pyt = np.concatenate(p_yt) if p_yt else np.array([])
    pys = np.concatenate(p_ys) if p_ys else np.array([])
    if len(pyt) and len(np.unique(pyt)) == 2:
        fpr, tpr, _ = roc_curve(pyt, pys)
        ax.plot(fpr, tpr, lw=2.8, color='black',
                label=f'pooled (all test, n={len(TEST_IN_RESULTS)}, AUC={roc_auc_score(pyt, pys):.3f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate (CI=positive)')
    ax.set_title(f'Inference ROC — {PRIMARY_TARGET}  (개별=VIZ, pooled=test 전부)')
    _handles, _labels = ax.get_legend_handles_labels()
    plt.tight_layout(); plt.show()

    # ===== Figure 1-Legend: legend 만 별도 Figure 로 완전히 분리 =====
    if _handles:
        figL = plt.figure(figsize=(3.8, 0.32 * len(_labels) + 0.6))
        figL.legend(_handles, _labels, loc='center', frameon=True,
                    title='dataset (AUC)', fontsize=8)
        plt.axis('off'); plt.tight_layout(); plt.show()

    # ===== Figure 2: 데이터셋별 AUC 막대 (VIZ_DS) — 별도 Figure =====
    sub_m = inf_metrics[(inf_metrics['target'] == PRIMARY_TARGET) &
                        (inf_metrics['dataset'].isin(VIZ_DS))].dropna(subset=['auc'])
    if len(sub_m):
        fig2, ax2 = plt.subplots(figsize=(max(6, 0.5 * len(sub_m)), 4.8))
        sns.barplot(data=sub_m, x='dataset', y='auc', ax=ax2, color='salmon')
        ax2.axhline(0.5, color='gray', linestyle='--', alpha=0.6)
        ax2.set_ylim(0, 1.05); ax2.set_title(f'Per-dataset AUC — {PRIMARY_TARGET}')
        ax2.tick_params(axis='x', rotation=60)
        plt.tight_layout(); plt.show()
    else:
        print('per-dataset AUC: 표시할 데이터 없음')
else:
    print('(no inference results — ROC 생략)')
